# CAIRLAB — full-data refit of the LLM (helper for `sentiment_pipeline_v4.ipynb`)
Runs **only** the step `llm_finetune(-1, tag="llm_qwen25_7b_v2", …)` of `sentiment_pipeline_v4.ipynb` §6b — the identical
function and hyper-parameters (base `Qwen/Qwen2.5-7B`, hand-written QLoRA, zero-initialised head, 2 epochs, seed 42), trained
on all 11,447 original tweets of `CAIRLAB.csv`, predicting `test.csv`. Used because the Colab GPU quota was exhausted.
Output: `/kaggle/working/llm_qwen25_7b_v2_full_test.npy` → copied into the notebook's cache folder (`FEAT_DIR`).

Kaggle settings: Accelerator **GPU T4**, Internet **On**.

In [ ]:
# Same transformers version as the Colab run of sentiment_pipeline_v4.ipynb; bitsandbytes for 4-bit loading.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.18.0", "bitsandbytes", "accelerate"], check=True)

In [ ]:
import os, gc, glob, time, random, warnings
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.use_deterministic_algorithms(True, warn_only=DEVICE.type == "cuda")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
transformers.logging.set_verbosity_error()
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/CAIRLAB.csv", recursive=True)[0])
FEAT_DIR, ONLY_CACHED, folds = "/kaggle/working", False, None    # fold = -1 below → folds are not used
train = pd.read_csv(f"{DATA_DIR}/CAIRLAB.csv", header=None, names=["ID", "Entity", "Sentiment", "Text"])
test = pd.read_csv(f"{DATA_DIR}/test.csv")
train["Text"] = train["Text"].fillna("").astype(str)
test["Text"] = test["Text"].fillna("").astype(str)
CLASSES = np.array(sorted(train.Sentiment.unique()))
train["y"] = train.Sentiment.map({c: i for i, c in enumerate(CLASSES)}).values
train["pos"] = train.groupby("ID").cumcount()
print("train", train.shape, "| test", test.shape, "| classes", CLASSES)

In [ ]:
def twitter_lm_prep(t: str) -> str:   # standard preprocessing for Twitter LMs
    return " ".join("@user" if w.startswith("@") and len(w) > 1 else "http" if w.startswith("http") else w
                    for w in t.split(" "))

In [ ]:
LLM_NAME, LLM_FOLD = "Qwen/Qwen2.5-7B", 0
LLM_CFG = dict(r=16, alpha=32, dropout=0.05, lr=1e-4, bs=16, max_len=128, smoothing=0.1)
LLM_RUNS = {   # tag: run-specific settings; the LLM prediction is the mean of the runs
    "llm_qwen25_7b":    dict(epochs=1, zero_head=False),   # first run: randomly initialised head, 1 epoch
    "llm_qwen25_7b_v2": dict(epochs=2, zero_head=True),    # second run: zero-initialised head, 2 epochs
}


class LoRALinear(torch.nn.Module):   # frozen (4-bit) linear layer + trainable low-rank update: y = W x + (alpha/r)·B(A(dropout(x)))
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        dev = next(base.parameters()).device
        self.base, self.scale, self.drop = base, alpha / r, torch.nn.Dropout(dropout)
        self.A = torch.nn.Linear(base.in_features, r, bias=False, device=dev, dtype=torch.float32)
        self.B = torch.nn.Linear(r, base.out_features, bias=False, device=dev, dtype=torch.float32)
        torch.nn.init.kaiming_uniform_(self.A.weight, a=5 ** 0.5)
        torch.nn.init.zeros_(self.B.weight)                      # adapter starts as a no-op
    def forward(self, x):
        return self.base(x) + self.B(self.A(self.drop(x))) * self.scale

def llm_finetune(fold, tag, model_name=LLM_NAME, r=16, alpha=32, dropout=0.05, lr=1e-4, epochs=1, bs=16,
                 max_len=128, smoothing=0.1, zero_head=True, seed=SEED):
    # fold >= 0: train on the original tweets of the other folds, predict the original tweets of `fold` + test.
    # fold = -1: train on all original tweets, predict test only. Results are cached in FEAT_DIR.
    suffix = f"fold{fold}" if fold >= 0 else "full"
    f_test, f_val, f_rows = (f"{FEAT_DIR}/{tag}_{suffix}_{s}.npy" for s in ("test", "valid", "valid_rows"))
    if os.path.exists(f_test) and (fold < 0 or (os.path.exists(f_val) and os.path.exists(f_rows))):
        print(f"loaded cache {tag} {suffix}")
        return ((np.load(f_val), np.load(f_rows)) if fold >= 0 else (None, None)), np.load(f_test)
    if ONLY_CACHED:
        return None
    seed_everything(seed)
    tr_rows, va_rows = folds[fold] if fold >= 0 else (np.arange(len(train)), np.array([], dtype=int))
    tr_df = train.iloc[tr_rows]; tr_df = tr_df[tr_df.pos == 0]
    va_df = train.iloc[va_rows]; va_df = va_df[va_df.pos == 0]
    print(f"[{tag} {suffix}] train (original tweets) {len(tr_df)} | hold-out {len(va_df)}", flush=True)

    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    load_kw = {}
    if DEVICE.type == "cuda":
        from transformers import BitsAndBytesConfig
        load_kw = dict(quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                              bnb_4bit_use_double_quant=True,
                                                              bnb_4bit_compute_dtype=torch.float16),
                       dtype=torch.float16, device_map={"": 0})
    t0 = time.time()
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CLASSES), **load_kw)
    model.config.pad_token_id = tok.pad_token_id                 # the head reads the last non-padding token
    model.config.use_cache = False
    for prm in model.parameters(): prm.requires_grad = False
    for layer in model.model.layers:
        for block, names in ((layer.self_attn, ("q_proj", "k_proj", "v_proj", "o_proj")),
                             (layer.mlp, ("gate_proj", "up_proj", "down_proj"))):
            for n in names:
                setattr(block, n, LoRALinear(getattr(block, n), r, alpha, dropout))
    model.score.float().requires_grad_(True)                     # classification head, trained in fp32
    if zero_head:
        torch.nn.init.zeros_(model.score.weight)                 # start from uniform predictions (random head → huge logits)
    if DEVICE.type == "cuda":
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    else:
        model.to(DEVICE)
    trainable = [q for q in model.parameters() if q.requires_grad]
    print(f"  loaded in {time.time()-t0:.0f}s | trainable parameters {sum(q.numel() for q in trainable)/1e6:.1f}M", flush=True)

    def encode(df):
        if len(df) == 0: return []
        texts = [f"{e}: {twitter_lm_prep(t)}" for e, t in zip(df.Entity, df.Text)]
        return tok(texts, truncation=True, max_length=max_len, add_special_tokens=False)["input_ids"]

    def batches(ids, order, size):
        for i in range(0, len(order), size):
            idx = order[i:i + size]
            L = max(len(ids[j]) for j in idx)
            x = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
            m = torch.zeros((len(idx), L), dtype=torch.long)
            for row, j in enumerate(idx):
                x[row, :len(ids[j])] = torch.tensor(ids[j]); m[row, :len(ids[j])] = 1
            yield idx, x.to(DEVICE), m.to(DEVICE)

    def bucket_order(ids, rng, size, chunk=50):
        perm = rng.permutation(len(ids)); out = []
        for c in range(0, len(perm), size * chunk):
            part = sorted(perm[c:c + size * chunk], key=lambda j: len(ids[j]))
            out += [part[i:i + size] for i in range(0, len(part), size)]
        rng.shuffle(out)
        return [j for b in out for j in b]

    use_amp = DEVICE.type == "cuda"
    @torch.inference_mode()
    def predict(ids):
        model.eval(); P = np.zeros((len(ids), len(CLASSES)), dtype=np.float32)
        for idx, x, m in batches(ids, np.argsort([len(v) for v in ids], kind="stable"), 32):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=x, attention_mask=m).logits
            P[idx] = torch.softmax(logits.float(), -1).cpu().numpy()
        return P

    tr_ids, va_ids, te_ids = encode(tr_df), encode(va_df), encode(test)
    ytr = torch.tensor(tr_df.y.values)
    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=0.0)
    steps = epochs * ((len(tr_ids) + bs - 1) // bs)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=smoothing)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_amp)
    rng = np.random.RandomState(seed)
    t0 = time.time(); step = 0
    for ep in range(epochs):
        model.train(); tot = n = 0
        for idx, x, m in batches(tr_ids, bucket_order(tr_ids, rng, bs), bs):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=x, attention_mask=m).logits
            loss = loss_fn(logits.float(), ytr[idx].to(DEVICE))
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
            tot += loss.item() * len(idx); n += len(idx); step += 1
            if step % 50 == 0 or step == steps:
                el = time.time() - t0
                print(f"  step {step}/{steps} loss {tot/n:.4f} | {el/60:.1f} min, ETA {el/step*(steps-step)/60:.1f} min", flush=True)
    if fold >= 0:
        Pva = predict(va_ids)
        print(f"  hold-out acc (fold {fold} original tweets) {(Pva.argmax(1) == va_df.y.values).mean():.4f} | {(time.time()-t0)/60:.1f} min", flush=True)
        np.save(f_val, Pva); np.save(f_rows, va_df.index.values)
    Pte = predict(te_ids); np.save(f_test, Pte)
    del model, opt
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return ((Pva, va_df.index.values) if fold >= 0 else (None, None)), Pte

In [ ]:
tag = "llm_qwen25_7b_v2"
(_, _), P_full_test = llm_finetune(-1, tag=tag, **(LLM_CFG | LLM_RUNS[tag]))
print(P_full_test.shape, P_full_test[:3].round(3))
print("saved:", sorted(f for f in os.listdir(FEAT_DIR) if f.endswith(".npy")))